In [22]:
import requests

# --- CONFIGURATION ---
# Replace <SERVER_IP> with the oracle address provided by your instructor,
# e.g. "http://192.168.1.50" or "http://10.0.0.5:80".
BASE_URL = "http://<SERVER_IP>"

HEADERS = {"Content-Type": "application/json"}


In [23]:
def query_oracle(pt_hex: str) -> str | None:
    """
    Encrypt a 16-hex-character plaintext with the shared-key oracle.
    Returns the ciphertext as an uppercase hex string, or None on error.
    """
    url = f"{BASE_URL}/api/encrypt"
    payload = {'plaintext': pt_hex}

    try:
        response = requests.post(url, headers=HEADERS, json=payload, timeout=15)

        # 429 = you have hit the per-IP daily query limit (default 1000/day).
        if response.status_code == 429:
            print("    ✗ Rate limit reached. Wait before sending more queries.")
            return None

        response.raise_for_status()
        result = response.json()
        ciphertext = result.get('ciphertext', '').upper()

        if len(ciphertext) == 16 and all(c in '0123456789ABCDEF' for c in ciphertext):
            return ciphertext

        print(f"    ✗ Unexpected response: {result}")
        return None

    except requests.exceptions.HTTPError as e:
        error_msg = response.json().get('detail', str(e)) if response.text else str(e)
        print(f"    ✗ HTTP {response.status_code}: {error_msg}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"    ✗ Connection error: {e}")
        return None
    except (ValueError, KeyError) as e:
        print(f"    ✗ Invalid JSON response: {e}")
        return None


In [24]:
test_pt = "FEDCBA9876543210"  # plaintext (16 hex chars)

print("--- Testing Oracle API Connection ---")
print(f"    Querying: 0x{test_pt}")
test_ct = query_oracle(test_pt)

if test_ct:
    print(f"Connection test successful! Received: 0x{test_ct}")
else:
    print("Connection test FAILED. Check BASE_URL and that the oracle is running.")
print("--- End Test ---")


--- Testing Oracle API Connection for cs24mtech12021 ---
    Querying cs24mtech12021: 0xFEDCBA9876543412
Connection test successful! Received: 0xAFDFFC4D02D83ECE
--- End Test ---
